# Create Bronze Layer Shortcuts

This notebook guides you through creating the Bronze layer shortcuts and fails
when a required shortcut is missing or unreadable.

## Shortcut Locations

| Source | Location | Count | Required |
|--------|----------|-------|----------|
| Eventhouse (streaming) | **Tables/cusn/** | 18 shortcuts | Yes: `03-streaming-to-silver` reads them |
| ADLS Gen2 (parquet) | **Files/** | 24 shortcuts | No: only the retained `02-historical-data-load` reads them |

`deploy/scripts/ensure_eventhouse_shortcuts.py` creates the 18 Eventhouse
shortcuts automatically when `streaming_shortcuts.enabled` is set; this
notebook is the manual path and the verification step.

**Note:** ADLS parquet shortcuts go in Files, Eventhouse shortcuts go in Tables.

In [ ]:
import os


def get_env(var_name, default=None):
    return os.environ.get(var_name, default)


# Configuration
LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
BRONZE_SCHEMA = "cusn"

# ADLS Gen2 settings
ADLS_ACCOUNT = "stdretail"
ADLS_CONTAINER = "supermarket"

# Eventhouse settings
EVENTHOUSE_DATABASE = "retail_eventhouse"

print(f"Bronze Schema: {BRONZE_SCHEMA}")
print(f"ADLS: https://{ADLS_ACCOUNT}.dfs.core.windows.net/{ADLS_CONTAINER}")
print(f"Eventhouse: {EVENTHOUSE_DATABASE}")

In [ ]:
# Create Bronze schema for Eventhouse shortcuts
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LAKEHOUSE_NAME}.{BRONZE_SCHEMA}")
print(f"✓ Schema '{BRONZE_SCHEMA}' created")

## Step 1 (optional, legacy loader only): Create ADLS Shortcuts in Files

Skip this step unless you run the retained `02-historical-data-load` notebook.
The supported historical setup writes Silver directly and does not read these.

In Fabric Portal:
1. Open your Lakehouse
2. Right-click **Files** → **New shortcut** → **Azure Data Lake Storage Gen2**
3. Create a shortcut for each folder listed below

These shortcuts point to parquet folders containing dimension and fact data.

In [ ]:
# Dimension folders (6)
dimension_tables = [
    "dim_geographies", "dim_stores", "dim_distribution_centers",
    "dim_trucks", "dim_customers", "dim_products"
]

# Fact folders (18)
fact_tables = [
    "fact_receipts", "fact_receipt_lines", "fact_store_inventory_txn",
    "fact_dc_inventory_txn", "fact_truck_moves", "fact_truck_inventory",
    "fact_foot_traffic", "fact_ble_pings", "fact_customer_zone_changes",
    "fact_marketing", "fact_online_order_headers", "fact_online_order_lines",
    "fact_payments", "fact_store_ops", "fact_stockouts",
    "fact_promotions", "fact_promo_lines", "fact_reorders"
]

all_parquet_tables = dimension_tables + fact_tables

print(f"Create these shortcuts in Files/ ({len(all_parquet_tables)} total):")
print(f"\nADLS Connection: https://{ADLS_ACCOUNT}.dfs.core.windows.net")
print(f"Container/Path: {ADLS_CONTAINER}/[folder_name]")
print()
for i, table in enumerate(all_parquet_tables, 1):
    print(f"  {i:2d}. {table}")

## Step 2 (required): Create Eventhouse Shortcuts in Tables

In Fabric Portal:
1. Right-click **Tables** → **New shortcut** → **Microsoft OneLake**
2. Select your Eventhouse → Select the database
3. Create a shortcut for each event table in the **cusn** schema

In [ ]:
# Event tables (18). Must equal EVENT_PAYLOADS in the stream generator;
# tests/deploy/test_bronze_shortcut_notebook.py enforces it.
event_tables = [
    # Transaction Events
    "receipt_created", "receipt_line_added", "payment_processed",
    # Inventory Events
    "inventory_updated", "stockout_detected", "reorder_triggered",
    # Customer Events
    "customer_entered", "customer_zone_changed", "ble_ping_detected",
    # Operational Events
    "truck_arrived", "truck_departed", "store_opened", "store_closed",
    "logistics_incident_started", "logistics_incident_resolved",
    # Marketing Events
    "ad_impression", "promotion_applied",
    # Omnichannel Events
    "online_order_created", "online_order_picked", "online_order_shipped",
    "online_order_line_added", "online_order_delivered"
]

print(f"Create these shortcuts in Tables/{BRONZE_SCHEMA}/ ({len(event_tables)} total):")
print(f"\nEventhouse Database: {EVENTHOUSE_DATABASE}")
print()
for i, table in enumerate(event_tables, 1):
    print(f"  {i:2d}. {table}")

## Step 3: Verify Shortcuts

The Eventhouse check fails the notebook when any required shortcut is missing.
The ADLS check only reports, because those shortcuts are optional.

In [ ]:
# Verify Files shortcuts (ADLS parquet; optional, legacy loader only)
from notebookutils import mssparkutils

try:
    file_shortcuts = [f.name for f in mssparkutils.fs.ls("Files/") if f.isDir]
except Exception as exc:  # noqa: BLE001 - optional legacy inputs
    file_shortcuts = []
    print(f"Files/ is not readable ({exc}); the legacy loader cannot run.")

missing_parquet = [t for t in all_parquet_tables if t not in file_shortcuts]
print(f"Files/ shortcuts: {len(all_parquet_tables) - len(missing_parquet)}/{len(all_parquet_tables)}")
if missing_parquet:
    print(
        "Optional ADLS shortcuts missing (needed only by 02-historical-data-load): "
        + ", ".join(missing_parquet)
    )
else:
    print("✓ All ADLS shortcuts created")

In [ ]:
# Verify Tables shortcuts (Eventhouse; required)
def missing_event_shortcuts(table_names):
    return [t for t in event_tables if t not in set(table_names)]


bronze_tables = spark.sql(f"SHOW TABLES IN {LAKEHOUSE_NAME}.{BRONZE_SCHEMA}").collect()
table_names = [t.tableName for t in bronze_tables]
missing_events = missing_event_shortcuts(table_names)
print(f"Tables/{BRONZE_SCHEMA}/ shortcuts: {len(event_tables) - len(missing_events)}/{len(event_tables)}")
if missing_events:
    raise RuntimeError(
        f"Required Eventhouse shortcuts are missing from Tables/{BRONZE_SCHEMA}: "
        + ", ".join(missing_events)
        + ". Create them (Step 2) or run deploy/scripts/ensure_eventhouse_shortcuts.py."
    )
print("✓ All Eventhouse shortcuts created")

## Step 4: Test Data Access

In [ ]:
# Test reading from shortcuts
print("Testing data access...\n")

# Required: every Eventhouse shortcut must be readable.
unreadable = []
for table in event_tables:
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{table}").limit(1).collect()
    except Exception as exc:  # noqa: BLE001 - collected and raised below
        unreadable.append(f"{table} ({type(exc).__name__})")
if unreadable:
    raise RuntimeError("Eventhouse shortcuts are not readable: " + ", ".join(unreadable))
print(f"✓ {len(event_tables)} Eventhouse shortcuts are readable")

# Optional: the legacy parquet input.
if not missing_parquet:
    try:
        df = spark.read.parquet("Files/dim_stores")
        print(f"✓ Files/dim_stores: {df.count()} rows")
    except Exception as e:
        print(f"Optional Files/dim_stores is not readable: {e}")

print("\nBronze layer ready!")